# Compile order commands and FIFO snapshots

Define a small adapter in Python, restore a book, apply order commands, and serve the result in the terminal. This notebook uses fixed raw JSON packets and needs no network connection.

The **envelope is defined here**: `type`, `symbol`, `sequence`, `timestamp_ns`, and either `orders` or `command`. The enclosed commands use the existing order-API schemas. `RestoreOrder` takes resting-order state in FIFO order, including iceberg quantities and creation timestamps. This is a custom recording format, not the server WebSocket feed's snapshot envelope.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "rust/crates/lobo_replay").is_dir())
sys.path.insert(0, str(root / "notebooks"))
from adapter_examples import wait_for_book
import pandas as pd
from IPython.display import display, HTML
from lobo.server import server_context

## Define the mapping

`CustomAdapter` compiles this declaration with Cranelift during construction. `OrderCommand` and `RestoreOrder` project fields directly into typed records and call the existing book API. Neither performs a second command/snapshot deserialization.

The JSON parser still builds a value tree before those projections. This does not demonstrate single-pass JSON-to-book decoding. Sequence and incoming-data errors remain runtime operations.

The instrument generator supplies the directory during construction. `book_policy="full"` selects user-map and hidden-quantity updates, enabling iceberg replenishment.

In [2]:
from lobo.replay.adapters import CustomAdapter, Protocol
from lobo.replay.adapters import expressions as le
from lobo.replay.adapters import models as lm


def protocol() -> Protocol:
    return Protocol(lm.Json(
        lm.Message(
            le.Field("type").eq("snapshot"),
            le.CheckSequence(le.Field("sequence"), reset=True),
            lm.Book(
                le.Field("symbol"),
                le.ForEach(le.Field("orders"), lm.RestoreOrder(le.Field())),
                snapshot=True,
                timestamp=le.Field("timestamp_ns"),
            ),
        ),
        lm.Message(
            le.Field("type").eq("update"),
            le.CheckSequence(le.Field("sequence")),
            lm.Book(
                le.Field("symbol"),
                lm.OrderCommand(le.Field("command")),
                timestamp=le.Field("timestamp_ns"),
            ),
        ),
    ))


def make_adapter(source):
    return CustomAdapter(
        definition, source, name="Order commands", symbol="BOOK", scope=["BOOK"],
        mode="live", level="l3",
        instruments=(lm.Instrument(symbol, 2, 0, book_policy="full") for symbol in ["BOOK"]),
    )


def queue_table(adapter):
    return pd.DataFrame(
        adapter.queue("BOOK", "sell", 10100, 10100),
        columns=["order_id", "price", "quantity", "timestamp_ns"],
    )


definition = protocol()

## Supply raw input

The snapshot's ask queue is **order 2, then order 3**, even though order 3 has the earlier creation timestamp. Restoration preserves the supplied FIFO order.

The updates execute order 2's peak, cancel five shares from order 3, modify the bid, add and remove another bid, preview a market order, and finally fill a market order. Replenishing the iceberg moves it behind order 3.

In [3]:
snapshot = b'''
{
  "type": "snapshot",
  "symbol": "BOOK",
  "sequence": 0,
  "timestamp_ns": 1000,
  "orders": [
    {
      "id": "00000000-0000-0000-0000-000000000001",
      "trader": "00000000-0000-0000-0000-000000000000",
      "side": "buy",
      "price": 10000,
      "quantity": 100,
      "created_at_ns": 100,
      "hidden_quantity": 0,
      "peak_quantity": 0
    },
    {
      "id": "00000000-0000-0000-0000-000000000002",
      "trader": "00000000-0000-0000-0000-000000000000",
      "side": "sell",
      "price": 10100,
      "quantity": 40,
      "created_at_ns": 200,
      "hidden_quantity": 80,
      "peak_quantity": 40
    },
    {
      "id": "00000000-0000-0000-0000-000000000003",
      "trader": "00000000-0000-0000-0000-000000000000",
      "side": "sell",
      "price": 10100,
      "quantity": 20,
      "created_at_ns": 100,
      "hidden_quantity": 0,
      "peak_quantity": 0
    }
  ]
}
'''

updates = [
    b'{"type":"update","symbol":"BOOK","sequence":1,"timestamp_ns":1100,"command":{"op":"execute","id":"00000000-0000-0000-0000-000000000002","quantity":40}}',
    b'{"type":"update","symbol":"BOOK","sequence":2,"timestamp_ns":1200,"command":{"op":"cancel","id":"00000000-0000-0000-0000-000000000003","quantity":5}}',
    b'{"type":"update","symbol":"BOOK","sequence":3,"timestamp_ns":1300,"command":{"op":"modify","id":"00000000-0000-0000-0000-000000000001","quantity":120}}',
    b'{"type":"update","symbol":"BOOK","sequence":4,"timestamp_ns":1400,"command":{"op":"add","order":{"type":"limit","id":"00000000-0000-0000-0000-000000000004","trader":"00000000-0000-0000-0000-000000000000","side":"buy","quantity":7,"price":9900}}}',
    b'{"type":"update","symbol":"BOOK","sequence":5,"timestamp_ns":1500,"command":{"op":"remove","id":"00000000-0000-0000-0000-000000000004"}}',
    b'{"type":"update","symbol":"BOOK","sequence":6,"timestamp_ns":1600,"command":{"op":"simulate","order":{"type":"market","id":"00000000-0000-0000-0000-000000000005","trader":"00000000-0000-0000-0000-000000000000","side":"buy","quantity":10}}}',
    b'{"type":"update","symbol":"BOOK","sequence":7,"timestamp_ns":1700,"command":{"op":"fill","order":{"type":"market","id":"00000000-0000-0000-0000-000000000006","trader":"00000000-0000-0000-0000-000000000000","side":"buy","quantity":10}}}',
]

## Restore the snapshot

The construction timer includes declaration conversion, validation and compilation, or cache reuse. It is not an isolated compiler benchmark.

In [4]:
started = perf_counter()
snapshot_adapter = make_adapter(lm.Source.packets([snapshot]))
construction_ms = (perf_counter() - started) * 1000
with snapshot_adapter:
    started = perf_counter()
    snapshot_adapter.start()
    snapshot_adapter.wait()
    completion_ms = (perf_counter() - started) * 1000
    before = queue_table(snapshot_adapter)
    assert before["order_id"].tolist() == [2, 3]
    display(before, pd.DataFrame(snapshot_adapter.levels("BOOK")))
print(f"Construction: {construction_ms:.3f} ms | Start to completion: {completion_ms:.3f} ms")

,order_id,price,quantity,timestamp_ns
0,2,10100,40,200
1,3,10100,20,100


,hidden,orders,price,quantity,side
0,0,1,10000,100,buy
1,80,2,10100,60,sell


Construction: 2.897 ms | Start to completion: 0.089 ms


## Apply the commands

The full recording is supplied in one call. The adapter consumes it through the same compiled program. The final real fill consumes ten shares from order 3; the preceding simulated command consumes none.

In [5]:
started = perf_counter()
command_adapter = make_adapter(lm.Source.packets([snapshot, *updates]))
construction_ms = (perf_counter() - started) * 1000
with command_adapter:
    started = perf_counter()
    command_adapter.start()
    command_adapter.wait()
    completion_ms = (perf_counter() - started) * 1000
    after = queue_table(command_adapter)
    status = command_adapter.status()
    assert after["order_id"].tolist() == [3, 2]
    assert after["quantity"].tolist() == [5, 40]
    display(after, pd.DataFrame(command_adapter.levels("BOOK")))
    print(f"Consumed {status['messages']} messages / {status['bytes']:,} bytes")
    command_adapter.simulate("buy", 10)
    report = command_adapter.simulation_report()
    assert report["filled"] == 10 and report["simulated"]
    assert queue_table(command_adapter).equals(after)
    display(pd.DataFrame(report["executions"]))
print(f"Construction (cached definition): {construction_ms:.3f} ms | Start to completion: {completion_ms:.3f} ms")

,order_id,price,quantity,timestamp_ns
0,3,10100,5,100
1,2,10100,40,200


,hidden,orders,price,quantity,side
0,0,1,10000,120,buy
1,40,2,10100,45,sell


Consumed 8 messages / 2,217 bytes


,price,quantity,sequence,simulated,timestamp_ns
0,10100,5,1,True,1700
1,10100,5,2,True,1700


Construction (cached definition): 0.154 ms | Start to completion: 0.103 ms


## Serve the books

Create a fresh session from the same declaration and recording, then attach it to the server. Open the link to inspect the book and use the simulation controls. The recording is finite; the completed book remains available until cleanup.

Run these cells individually while exploring. Run All reaches cleanup and closes the server.

In [6]:
adapter = make_adapter(lm.Source.packets([snapshot, *updates]))
terminal = server_context(adapters=[adapter], port=0)
adapter.wait()
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))
display(queue_table(adapter))

,order_id,price,quantity,timestamp_ns
0,3,10100,5,100
1,2,10100,40,200


Run when finished exploring.

In [7]:
terminal.close()

## Recorded performance result

The September 9, 2026 paired order-API benchmark measured **139.0 µs existing / 303.3 µs custom** for 200 commands, a **2.18×** paired ratio. This excludes compilation and construction. It is a saved benchmark result, not a timing from this notebook; streaming parity remains unmet.

See [validation and reproduction commands](../python/examples/VALIDATION.md).